# Fact country year

create FSI first

In [ ]:
import duckdb

DB_PATH = '../data/conflict_prediction.duckdb' 
con = duckdb.connect(DB_PATH)


try:

  con.execute("DROP TABLE IF EXISTS fsi_country_year")
# making tabkle fsi country year
  con.execute("""
CREATE TABLE fsi_country_year AS
SELECT
    country_code,
    CAST(year AS INT) AS year,
    total_score AS fsi_total_score,
    demographic_pressures,
    refugees_idps,
    group_grievance,
    human_flight,
    economic_inequality,
    economy,
    state_legitimacy,
    public_services,
    human_rights,
    security_apparatus,
    factionalized_elites,
    external_intervention
FROM fragile_states_index
WHERE country_code IS NOT NULL
  AND year IS NOT NULL
""")
except exception as e:
  print(e)

then acled

In [ ]:

try:
    con.execute("DROP TABLE IF EXISTS acled_country_year")
# creating table acled country year
    con.execute("""
CREATE TABLE acled_country_year AS
SELECT
    country_code,
    year,
    COUNT(*) AS acled_event_count,
    SUM(fatalities) AS acled_fatalities_sum,
    SUM(
        CASE
            WHEN event_type IN ('Battles', 'Violence against civilians', 'Explosions/Remote violence')
            THEN 1 ELSE 0
        END
    ) AS acled_violent_event_count,
    SUM(
        CASE
            WHEN event_type = 'Protests'
            THEN 1 ELSE 0
        END
    ) AS acled_protest_count
FROM fact_acled_event
GROUP BY country_code, year
ORDER BY country_code, year
""")
except exception as e:
  print(e)

print("acled_country_year created")
print(con.execute("SELECT * FROM acled_country_year LIMIT 10").fetchdf())

then gdelt

In [ ]:
try:
    con.execute("DROP TABLE IF EXISTS gdelt_country_year")
# gdelt country year
    con.execute("""
CREATE TABLE gdelt_country_year AS
SELECT
    country_code,
    year,
    COUNT(*) AS gdelt_event_count,
    AVG(avgtone) AS gdelt_avg_tone,
    AVG(goldsteinscale) AS gdelt_avg_goldstein,
    SUM(nummentions) AS gdelt_total_mentions,
    SUM(numsources) AS gdelt_total_sources
FROM fact_gdelt_event
GROUP BY country_code, year
ORDER BY country_code, year
""")
except exception as e:
  print(e)

print("gdelt_country_year created")
print(con.execute("SELECT COUNT(*) AS rows FROM gdelt_country_year").fetchdf())
print(con.execute("SELECT * FROM gdelt_country_year LIMIT 10").fetchdf())

then all of it together

In [ ]:
try:
    con.execute("DROP TABLE IF EXISTS fact_country_year")
# fact country year aswell
    con.execute("""
CREATE TABLE fact_country_year AS
WITH all_years AS (
    SELECT country_code, year FROM acled_country_year
    UNION
    SELECT country_code, year FROM gdelt_country_year
    UNION
    SELECT country_code, year FROM military_country_year
    UNION
    SELECT country_code, year FROM fsi_country_year
)
SELECT
    country_code,
    country_name,
    region,
    income_group,
    lending_category,
    year,

    COALESCE(acled_event_count, 0) AS acled_event_count,
    COALESCE(acled_fatalities_sum, 0) AS acled_fatalities_sum,
    COALESCE(acled_violent_event_count, 0) AS acled_violent_event_count,
    COALESCE(acled_protest_count, 0) AS acled_protest_count,

    COALESCE(gdelt_event_count, 0) AS gdelt_event_count,
    gdelt_avg_tone,
    gdelt_avg_goldstein,
    COALESCE(gdelt_total_mentions, 0) AS gdelt_total_mentions,
    COALESCE(gdelt_total_sources, 0) AS gdelt_total_sources,

    military_spending_current_usd,

    fsi_total_score,
    demographic_pressures,
    refugees_idps,
    group_grievance,
    human_flight,
    economic_inequality,
    economy,
    state_legitimacy,
    public_services,
    human_rights,
    security_apparatus,
    factionalized_elites,
    external_intervention

FROM all_years
JOIN dim_country USING (country_code)
LEFT JOIN acled_country_year USING (country_code, year)
LEFT JOIN gdelt_country_year USING (country_code, year)
LEFT JOIN military_country_year USING (country_code, year)
LEFT JOIN fsi_country_year USING (country_code, year)
ORDER BY country_code, year
""")
except exception as e:
  print(e)
  
print("fact_country_year created")
print(con.execute("SELECT COUNT(*) AS rows FROM fact_country_year").fetchdf())
print(con.execute("SELECT * FROM fact_country_year LIMIT 10").fetchdf())